# Evaluasi Model Klasifikasi Framework Web

Jalankan *cell* di bawah ini. Anda akan diminta untuk **mengunggah (upload)** file `dataset_train.csv` dan `dataset_test.csv` melalui tombol *Choose Files* yang akan muncul.

In [ ]:
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Menyediakan tombol trigger untuk upload file di Google Colab
from google.colab import files
print("Silakan klik 'Choose Files' dan pilih dataset_train.csv serta dataset_test.csv secara bersamaan:")
uploaded = files.upload()

# Deteksi otomatis mana file train dan test dari nama file yang diunggah
nama_file_diunggah = list(uploaded.keys())
train_file = next(f for f in nama_file_diunggah if 'train' in f.lower())
test_file = next(f for f in nama_file_diunggah if 'test' in f.lower())

# Memuat dataset ke dalam Pandas DataFrame
df_train = pd.read_csv(io.BytesIO(uploaded[train_file]))
df_test = pd.read_csv(io.BytesIO(uploaded[test_file]))

print(f"\nBerhasil memuat data training dari: {train_file}")
print(f"Berhasil memuat data testing dari: {test_file}")

# Memisahkan fitur dan target
X_train = df_train.drop(columns=['Y_framework', 'repository'])
y_train = df_train['Y_framework']

X_test = df_test.drop(columns=['Y_framework', 'repository'])
y_test = df_test['Y_framework']

# Mapping Framework
framework_mapping = {
    1: 'Django', 2: 'FastAPI', 3: 'Ruby on Rails',
    4: 'Next.js', 5: 'Flask', 6: 'Spring Boot',
    7: 'Laravel', 8: 'Symfony', 9: 'Express.js'
}
labels = [framework_mapping[i] for i in range(1, 10)]

print("\nDataset Training Shape:", X_train.shape)
print("Dataset Testing Shape:", X_test.shape)
df_train.head()

In [ ]:
# Melatih Model
print("Melatih model Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

# Melakukan Prediksi
y_pred = rf.predict(X_test)
print("Selesai melatih dan melakukan prediksi pada data testing!")

## 1. Metrik Evaluasi Utama (Tabel)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
macro_precision = precision_score(y_test, y_pred, average='macro')
macro_recall = recall_score(y_test, y_pred, average='macro')
macro_f1 = f1_score(y_test, y_pred, average='macro')

metrics_df = pd.DataFrame({
    'Metrik': ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1-Score'],
    'Nilai': [accuracy, macro_precision, macro_recall, macro_f1]
})

display(metrics_df)

## 2. Classification Report (Tabel Kinerja per Kelas)

In [ ]:
report_dict = classification_report(y_test, y_pred, target_names=labels, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

display(report_df)

## 3. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.index.name = 'Actual Framework'
cm_df.columns.name = 'Predicted Framework'

print("Tabel Confusion Matrix:")
display(cm_df)

print("\nGrafik Heatmap:")
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Framework')
plt.ylabel('Actual Framework')
plt.xticks(rotation=45)
plt.show()

## 4. Feature Importance (Karakteristik Paling Berpengaruh)

In [ ]:
importances = rf.feature_importances_
features = X_train.columns
clean_features = [f.split('_', 1)[1] if '_' in f else f for f in features]

importance_df = pd.DataFrame({
    'Karakteristik Proyek': clean_features,
    'Tingkat Kepentingan': importances
}).sort_values(by='Tingkat Kepentingan', ascending=False)

print("Tabel Feature Importance:")
display(importance_df)

print("\nGrafik Feature Importance:")
plt.figure(figsize=(10, 6))
sns.barplot(x='Tingkat Kepentingan', y='Karakteristik Proyek', data=importance_df, palette='viridis')
plt.title('Feature Importances in Random Forest')
plt.show()